# ODER Linguistic Spectral-Admissibility Verification Notebook

This notebook does one narrow job: it checks whether the tanh smooth-access baseline used in linguistic ODER leaves a recoverable inverse-$\gamma$ signature under finite resolution.

The test compares tanh against calibrated bounded saturating alternatives: logistic, Gompertz, stretched exponential, and hyperbolic envelopes. The alternatives are placed on the same saturation timescale, passed through the same inverse decoder, and tested under declared noise conditions.

The point is not to show that human language data are tanh-shaped. This is a theory-side verification artifact. It asks whether the analytic tanh baseline has operational consequences in synthetic traces, whether those consequences survive finite-resolution noise, and whether adversarial foils become confusable only by moving close to tanh below the declared measurement floor.

The notebook therefore supports the theory-side claim that spectral admissibility is not an empty formal condition. It does not constitute empirical confirmation from human ERP, self-paced reading, or eye-tracking data.


## Version Lock

```text
Artifact: ODER_Linguistic_Spectral_Admissibility_verification_artifact_v1
Version: 1.0
Run mode: theory-side verification
Analysis preset: LINGUISTIC_SPECTRAL_ADMISSIBILITY_V1
Scientific status: frozen
Primary noise model: heteroscedastic
Declared reference noise level: sigma = 0.04
Noise levels: 0.0, 0.005, 0.01, 0.02, 0.04, 0.08, 0.16
Trace regimes: short, medium, long
Five-class samples per class: 300
Pairwise samples per class: 400
Boundary samples per class: 500
Adversarial classifier samples per class: 600
Master seed: 42
Allowed changes: comments, formatting, dependency bootstrap, output organization, manifest export
Disallowed changes: retrieval law, envelope definitions, inverse map, feature definitions, noise grid, classifiers, seeds, thresholds, headline criteria
```


## Validation Scope and PASS Criteria

| Check | PASS criterion |
| --- | --- |
| Calibration | every envelope reaches S(3.0) = 0.95 within 1e-6 |
| Primary five-class discriminability | heteroscedastic accuracy at sigma=0.04 is at least 0.80 for short, medium, and long regimes |
| Pairwise foil discriminability | tanh-vs-foil accuracy at sigma=0.04 is at least 0.90 for every calibrated foil |
| Adversarial near-equivalence | adversarial Gompertz max deviation is below the sigma=0.04 measurement floor |
| Classifier robustness for adversarial foil | all classifier accuracies remain at or below 0.65 when the adversarial foil is within the noise floor |
| Trajectory-RMS boundary | tanh-vs-Gompertz accuracy is at least 0.95 once RMS deviation exceeds sigma=0.04 |


## Reader Map

1. Environment bootstrap and fixed verification configuration
2. Envelope families, noise models, inverse decoder, and feature maps
3. Calibration to a common saturation reference
4. Finite-resolution discriminability under noise
5. Pairwise calibrated-foil tests
6. Adversarial Gompertz follow-up
7. Trajectory-RMS boundary characterization
8. Manifest export and final verification report


## 0. Environment Bootstrap

This cell checks the packages required by the spectral-admissibility verification notebook and installs missing packages into the active notebook kernel when `INSTALL_MISSING=True`. The printed install command uses the current kernel executable and is not part of the scientific preset.


In [ ]:
# Dependency bootstrap.
import importlib.util
import subprocess
import sys

INSTALL_MISSING = True
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
}

missing = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(import_name) is None]

if missing:
    print("Dependency bootstrap: installing missing packages into this kernel environment.")
    print("Required packages:", ", ".join(missing))
    print("Install command:")
    print(sys.executable, "-m", "pip", "install", *missing)
    if INSTALL_MISSING:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
        print("Install complete. Restart the kernel if imports still fail, then run from the top.")
    else:
        raise ModuleNotFoundError(
            "Missing notebook dependencies. Set INSTALL_MISSING=True in this cell, "
            "or install the printed packages into the active kernel environment."
        )
else:
    print("Dependency check passed: required spectral-admissibility packages are available.")


### 1. Setup and Fixed Verification Configuration

Runtime output locations may change. The retrieval law, envelope definitions, inverse map, noise grid, seeds, and PASS thresholds are part of the frozen verification preset.


In [ ]:
import numpy as np
import os, json, platform
from pathlib import Path
from datetime import datetime, timezone
import matplotlib.pyplot as plt
from scipy.optimize import brentq, differential_evolution
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

# Master seed. Section-level seeds are derived from this for reproducibility
# without forcing every cell to share the same random state.
MASTER_SEED = 42

# Plot defaults
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

ARTIFACT_NAME = "ODER_Linguistic_Spectral_Admissibility_verification_artifact_v1"
ARTIFACT_VERSION = "1.0"
ANALYSIS_PRESET = "LINGUISTIC_SPECTRAL_ADMISSIBILITY_V1"
RUN_MODE = "theory-side verification"
OUTPUT_DIR = Path(os.getenv("ODER_SPECTRAL_OUTPUT_ROOT", "./oder_spectral_outputs"))
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_NOISE_MODEL = "heteroscedastic"
REFERENCE_NOISE = 0.04
PASS_THRESHOLDS = {
    "calibration_abs_error_max": 1e-6,
    "five_class_accuracy_min_at_reference_noise": 0.80,
    "pairwise_accuracy_min_at_reference_noise": 0.90,
    "adversarial_max_deviation_max": REFERENCE_NOISE,
    "adversarial_classifier_accuracy_max": 0.65,
    "rms_boundary_accuracy_min_above_noise_floor": 0.95,
}

print(f"Artifact: {ARTIFACT_NAME}")
print(f"Version: {ARTIFACT_VERSION}")
print(f"Analysis preset: {ANALYSIS_PRESET}")
print(f"Output directory: {OUTPUT_DIR}")




## 1. Envelope families

This section defines five bounded, monotone saturating envelopes. The tanh envelope is the spectrally admissible extremal case used by linguistic ODER. The other families -- logistic, Gompertz, stretched exponential, and hyperbolic -- serve as bounded saturating foils.

Each envelope maps $[0,\infty)$ to $[0,1)$ with $S(0)=0$. The families differ in onset, curvature, and approach to saturation. Section 7.1 calibrates them to a common 95% retrieval time so that classification cannot be driven by trivial endpoint or timescale differences.


In [ ]:
def tanh_envelope(tau, tau_char=1.0):
    """The spectrally-admissible extremal saturator (Section 7)."""
    return np.tanh(tau / tau_char)


def logistic_envelope(tau, k=1.5, tau_mid=1.0):
    """Standard logistic, rescaled to S(0) = 0."""
    L = 1.0 / (1.0 + np.exp(-k * (tau - tau_mid)))
    L0 = 1.0 / (1.0 + np.exp(k * tau_mid))
    return (L - L0) / (1.0 - L0)


def gompertz_envelope(tau, c=1.0, eta=0.5):
    """Gompertz-class saturator: 1 - exp(-eta * (exp(c*tau) - 1) / c).

    Parameter c controls the asymmetry; eta sets the late-time decay rate.
    Defaults give a curve qualitatively similar to logistic but with a
    different early-vs-late shape balance.
    """
    if abs(c) < 1e-6:
        return 1.0 - np.exp(-eta * tau)
    return 1.0 - np.exp(-eta * (np.exp(c * tau) - 1.0) / c)


def stretched_exp_envelope(tau, beta=0.7, tau_s=1.0):
    """Stretched (Kohlrausch) exponential. beta < 1 produces a heavy early
    rise and slower late approach to saturation."""
    return 1.0 - np.exp(-(tau / tau_s) ** beta)


def hyperbolic_envelope(tau, alpha=1.0):
    """Simple hyperbolic saturator."""
    return 1.0 - 1.0 / (1.0 + alpha * tau)


# Registry. Names are used as keys throughout the notebook.
ENVELOPES = {
    'tanh': tanh_envelope,
    'logistic': logistic_envelope,
    'gompertz': gompertz_envelope,
    'stretched_exp': stretched_exp_envelope,
    'hyperbolic': hyperbolic_envelope,
}



## 2. Noise models

This section defines two noise structures.

Additive Gaussian noise provides the baseline case. It applies a constant noise scale across the trajectory and makes the measurement floor easy to interpret.

Heteroscedastic noise scales with the local retrieval rate. This approximates a common measurement condition in neural and behavioral data: uncertainty increases where the signal changes most rapidly. For that reason, the heteroscedastic model is treated as the more demanding finite-resolution stress test: it asks whether the trajectory remains recoverable when noise is largest near the parts of the trace where the signal changes most.


In [ ]:
def add_gaussian_noise(S, sigma, rng):
    """Additive Gaussian noise."""
    return S + rng.normal(0, sigma, size=S.shape)


def add_heteroscedastic_noise(S, sigma_base, rng):
    """Noise scales with local retrieval rate dS/dτ.

    Floors at sigma_base/4 to avoid degenerate zero-noise regions in the
    saturated tail.
    """
    dS = np.abs(np.gradient(S))
    sigma_local = sigma_base * (0.25 + dS / (np.max(dS) + 1e-9))
    return S + rng.normal(0, sigma_local)


NOISE_MODELS = {
    'gaussian': add_gaussian_noise,
    'heteroscedastic': add_heteroscedastic_noise,
}



## 3. Inverse decoder

The retrieval law in the framework paper is

$$
\frac{dS_{\mathrm{obs}}}{d\tau}
=
\gamma(\tau)
\left(S_{\max}-S_{\mathrm{obs}}(\tau)\right)
\tanh\left(\frac{\tau}{\tau_{\mathrm{char}}}\right).
$$

Solving for $\gamma(\tau)$ gives the inverse decoder:

$$
\gamma(\tau)
=
\frac{dS_{\mathrm{obs}}/d\tau}
{\left(S_{\max}-S_{\mathrm{obs}}(\tau)\right)
\tanh\left(\tau/\tau_{\mathrm{char}}\right)}.
$$

The inversion is evaluated only on the admissible region of the trace. This is the region where both the onset regulator and the remaining retrieval gap are bounded away from zero:

$$
\tanh(\tau/\tau_{\mathrm{char}}) \ge \varepsilon_\tau,
\qquad
S_{\max}-S_{\mathrm{obs}}(\tau) \ge \varepsilon_S.
$$

Outside this region, the denominator becomes small and recovery of $\gamma(\tau)$ is ill-conditioned.

The discriminator applies this inverse decoder to every trace using the tanh calibration for $\tau_{\mathrm{char}}$. Tanh-generated traces should therefore recover relatively stable $\gamma(\tau)$. Non-tanh traces are forced through the same decoder and produce recovered $\gamma(\tau)$ signatures with family-specific shape structure.


In [ ]:
def inverse_gamma(tau, S, tau_char, S_max=1.0,
                  eps_tau=0.15, eps_S=0.05):
    """Recover γ(τ) inside the admissible inference region.

    Returns
    -------
    tau_admissible : ndarray
        τ values inside the admissible region.
    gamma_admissible : ndarray
        Recovered γ at those points (NaN where unstable).
    mask : ndarray of bool
        Boolean admissibility mask over the input τ grid.
    """
    onset_term = np.tanh(tau / tau_char)
    gap_term = S_max - S
    dS_dtau = np.gradient(S, tau)

    mask = (onset_term >= eps_tau) & (gap_term >= eps_S)
    valid = mask & (onset_term > 0) & (gap_term > 0)

    gamma = np.full_like(tau, np.nan)
    gamma[valid] = dS_dtau[valid] / (gap_term[valid] * onset_term[valid])

    return tau[mask], gamma[mask], mask



## 4. Featurization

This section defines two feature sets for the recovered $\gamma(\tau)$ profile.

**Five-feature set.** This is the default feature set used in the headline discriminability tests. It includes the mean of $\gamma$, coefficient of variation, normalized linear slope, normalized curvature, and skew. These features compress the recovered profile into shape descriptors. They reduce dependence on absolute scale and emphasize whether recovered $\gamma(\tau)$ is approximately stable, as expected under the tanh smooth-access baseline, or time-structured, as expected when non-tanh envelopes are forced through the tanh inverse decoder.

**Sixteen-feature set.** This feature set is used as a robustness check in Section 8.3. It includes the five $\gamma$-shape features, eight derivative samples at fixed $\tau$ points, the half-life, two curvature integrals, and the time-to-90% / time-to-50% shape ratio. This richer representation tests whether discriminability survives when the classifier is given more of the trajectory geometry rather than only the compressed five-feature summary.


In [ ]:
def gamma_features_5(tau_adm, gamma_adm):
    """Five shape features of the recovered γ(τ) profile.

    Returns NaN vector if the profile is too short or fully NaN.
    """
    if len(gamma_adm) < 4 or np.all(np.isnan(gamma_adm)):
        return np.full(5, np.nan)
    g = gamma_adm[~np.isnan(gamma_adm)]
    t = tau_adm[~np.isnan(gamma_adm)]
    if len(g) < 4:
        return np.full(5, np.nan)

    mean_g = np.mean(g)
    std_g = np.std(g)
    cv = std_g / (np.abs(mean_g) + 1e-9)
    slope = np.polyfit(t, g, 1)[0] / (np.abs(mean_g) + 1e-9)
    curvature = np.polyfit(t, g, 2)[0] / (np.abs(mean_g) + 1e-9) if len(g) >= 5 else 0.0
    skew = (np.mean(((g - mean_g) / std_g) ** 3) if std_g > 1e-9 else 0.0)

    return np.array([mean_g, cv, slope, curvature, skew])


def gamma_features_16(tau, S, tau_char):
    """Richer 16-feature representation: γ-shape + trajectory shape.

    Used in the adversarial robustness check (Section 8.3).
    """
    feats = []

    # 5 γ-shape features
    ta, ga, _ = inverse_gamma(tau, S, tau_char)
    feats.extend(gamma_features_5(ta, ga).tolist())

    # 8 derivative samples at fixed τ points
    if len(S) > 5:
        dS = np.gradient(S, tau)
        sample_taus = np.linspace(tau[0]*1.5, tau[-1]*0.95, 8)
        feats.extend(np.interp(sample_taus, tau, dS).tolist())

        # Half-life
        if S[-1] > 0.5:
            feats.append(tau[np.argmin(np.abs(S - 0.5))])
        else:
            feats.append(tau[-1])

        # Curvature integrals
        d2S = np.gradient(dS, tau)
        feats.append(float(np.mean(d2S)))
        feats.append(float(np.mean(d2S * tau)))

        # Time-to-90% / time-to-50% ratio
        idx50 = np.argmin(np.abs(S - 0.5))
        idx90 = np.argmin(np.abs(S - 0.9))
        if idx90 > idx50 and idx50 > 0:
            feats.append(tau[idx90] / tau[idx50])
        else:
            feats.append(np.nan)
    else:
        feats.extend([np.nan] * 12)

    return np.array(feats)



## 5. Trace generation

This section generates noisy retrieval traces from a named envelope family.

The underlying envelope is bounded and monotone. The observed trace is clipped to \([0,1]\) to preserve retrieval bounds, but no per-sample monotonicity constraint is imposed after noise is added. Enforcing monotonicity on each noisy trace would remove part of the measurement error the notebook is designed to test.

The classifier therefore sees noisy bounded observations, not cleaned monotone reconstructions.


In [ ]:
def generate_trace(envelope_fn, envelope_params, tau,
                    noise_model, noise_level, rng):
    """Generate a single noisy trace from a given envelope.

    Parameters
    ----------
    envelope_fn : callable
        One of the envelope functions from Section 1, OR a callable taking τ alone.
    envelope_params : dict or None
        Parameters for envelope_fn. Pass None if the envelope is already
        partially applied.
    tau : ndarray
        Time grid.
    noise_model : str
        Key in NOISE_MODELS, or 'none' for clean traces.
    noise_level : float
        σ for the noise process.
    rng : numpy.random.Generator

    Returns
    -------
    S_noisy : ndarray
    S_clean : ndarray
    """
    if envelope_params is None:
        S_clean = envelope_fn(tau)
    else:
        S_clean = envelope_fn(tau, **envelope_params)

    if noise_model == 'none' or noise_level == 0.0:
        return np.clip(S_clean, 0.0, 1.0), S_clean

    noise_fn = NOISE_MODELS[noise_model]
    S = noise_fn(S_clean, noise_level, rng)
    return np.clip(S, 0.0, 1.0), S_clean



## 6. Discriminability framework

This section defines the reusable discriminability test.

For a given set of envelope families, the function generates noisy traces, applies the tanh inverse decoder, extracts recovered $\gamma(\tau)$ features, and evaluates classifier performance under stratified cross-validation. The same function is used for both pairwise tanh-vs-foil tests and multi-class envelope classification.

The returned outputs are accuracy, confusion matrix, and optional AUC. These outputs measure whether trajectory geometry remains recoverable after noise and inverse decoding, not whether a classifier can identify a family label in the abstract.


In [ ]:
def run_discriminability(envelope_specs, tau, tau_char_decoder,
                          noise_model, noise_level,
                          n_per_class, rng,
                          featurizer=gamma_features_5,
                          classifier=None,
                          cv_seed=0):
    """Run a discriminability experiment.

    Parameters
    ----------
    envelope_specs : list of (label_name, envelope_fn, envelope_params)
        Each tuple defines one class. For envelopes from ENVELOPES with their
        calibrated parameters, use (name, ENVELOPES[name], CALIB[name]).
    tau : ndarray
        Time grid.
    tau_char_decoder : float
        τ_char to use in the inverse decoder. Use the calibrated tanh value
        so the decoder operates "as if" all traces were tanh.
    noise_model, noise_level : noise specification.
    n_per_class : int
        Samples per envelope class.
    rng : numpy.random.Generator
    featurizer : callable
        Either gamma_features_5 (operates on tau_adm, gamma_adm) or
        gamma_features_16 (operates on tau, S, tau_char).
    classifier : sklearn estimator
        Defaults to LDA.

    Returns
    -------
    dict with keys: accuracy, confusion, labels, n_used, n_failed, auc (binary only)
    """
    if classifier is None:
        classifier = LinearDiscriminantAnalysis()

    labels = [spec[0] for spec in envelope_specs]
    X, y = [], []
    n_failed = 0

    for class_idx, (_, env_fn, env_params) in enumerate(envelope_specs):
        for _ in range(n_per_class):
            S_noisy, _ = generate_trace(env_fn, env_params, tau,
                                          noise_model, noise_level, rng)
            if featurizer is gamma_features_5:
                ta, ga, _ = inverse_gamma(tau, S_noisy, tau_char_decoder)
                feats = gamma_features_5(ta, ga)
            else:
                feats = featurizer(tau, S_noisy, tau_char_decoder)

            if np.any(np.isnan(feats)):
                n_failed += 1
                continue
            X.append(feats)
            y.append(class_idx)

    X = np.array(X)
    y = np.array(y)
    if len(X) < 5 * len(envelope_specs):
        return {'error': 'too few valid samples', 'n_failed': n_failed}

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cv_seed)
    y_pred = cross_val_predict(classifier, X, y, cv=cv)

    result = {
        'accuracy': float(np.mean(y_pred == y)),
        'confusion': confusion_matrix(y, y_pred, labels=range(len(labels))),
        'labels': labels,
        'n_used': len(X),
        'n_failed': n_failed,
    }

    # Add AUC for binary problems
    if len(labels) == 2 and hasattr(classifier, 'predict_proba'):
        try:
            y_proba = cross_val_predict(classifier, X, y, cv=cv,
                                          method='predict_proba')
            result['auc'] = float(roc_auc_score(y, y_proba[:, 1]))
        except Exception:
            result['auc'] = None

    return result



# 7. Verification Demonstrations

The preceding sections define the reusable machinery. The cells below run the baseline discriminability tests, pairwise foil comparisons, adversarial Gompertz search, and trajectory-RMS boundary characterization.


## 7.1 Calibrate envelopes to a common saturation reference

Each envelope family has its own shape parameters. We calibrate all five so that they reach 95% saturation at

$$
\tau = 3.0.
$$

This puts the families on the same retrieval timescale. Without this step, the discriminator could separate curves by endpoint timing rather than by onset, curvature, saturation approach, or recovered $\gamma(\tau)$ structure. The test would then measure trivial scale differences rather than nontrivial trajectory geometry.


In [ ]:
def _calibrate_to_95_at_tau(envelope_fn, target_tau=3.0, target_val=0.95):
    """Find the parameter for each envelope so that S(target_tau) = target_val."""

    def find_param(p_low, p_high, param_name, **fixed):
        def err(p):
            kw = {param_name: p, **fixed}
            return envelope_fn(target_tau, **kw) - target_val
        return brentq(err, p_low, p_high)

    if envelope_fn is tanh_envelope:
        return {'tau_char': target_tau / np.arctanh(target_val)}
    if envelope_fn is logistic_envelope:
        return {'k': find_param(0.5, 10.0, 'k', tau_mid=1.0), 'tau_mid': 1.0}
    if envelope_fn is gompertz_envelope:
        return {'c': find_param(0.05, 5.0, 'c'), 'eta': 0.5}
    if envelope_fn is stretched_exp_envelope:
        return {'beta': 0.7, 'tau_s': find_param(0.01, 20.0, 'tau_s', beta=0.7)}
    if envelope_fn is hyperbolic_envelope:
        return {'alpha': find_param(0.1, 100.0, 'alpha')}


CALIB = {name: _calibrate_to_95_at_tau(fn) for name, fn in ENVELOPES.items()}

print("Calibrated parameters (95% saturation at tau=3.0):")
for name, params in CALIB.items():
    val_at_1 = ENVELOPES[name](1.0, **params)
    val_at_3 = ENVELOPES[name](3.0, **params)
    print(f"  {name:15s}: {params}")
    print(f"  {'':15s}  S(1)={val_at_1:.3f}  S(3)={val_at_3:.3f}")



## 7.2 Envelope shapes and recovered γ signatures

This section shows two diagnostic views of the calibrated clean traces.

The left panel shows the envelopes themselves. All families reach the same 95% retrieval point at $\tau = 3.0$, but they reach that point along different trajectories.

The right panel shows the recovered $\gamma(\tau)$ profile produced when each trace is passed through the tanh inverse decoder. Tanh traces recover an approximately stable $\gamma(\tau)$ profile. Non-tanh traces produce structured time variation because their trajectory geometry is incompatible with the tanh decoder.

The discriminator operates on these recovered $\gamma(\tau)$ signatures rather than on endpoint timing.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
tau_dense = np.linspace(0.01, 4.0, 200)
tau_char_tanh = CALIB['tanh']['tau_char']
colors = {'tanh': 'C0', 'logistic': 'C1', 'gompertz': 'C2',
          'stretched_exp': 'C3', 'hyperbolic': 'C4'}

# Left panel: envelopes
ax = axes[0]
for name, fn in ENVELOPES.items():
    S = fn(tau_dense, **CALIB[name])
    ax.plot(tau_dense, S, label=name, color=colors[name], lw=2)
ax.axhline(0.95, ls=':', color='gray', alpha=0.5)
ax.axvline(3.0, ls=':', color='gray', alpha=0.5)
ax.set_xlabel(r'observer-indexed retrieval time $\tau$')
ax.set_ylabel(r'$S(\tau)$')
ax.set_title('Calibrated envelopes (all reach 95% at τ=3)')
ax.legend(fontsize=9, loc='lower right')

# Right panel: γ-recovery
ax = axes[1]
for name, fn in ENVELOPES.items():
    S = fn(tau_dense, **CALIB[name])
    ta, ga, _ = inverse_gamma(tau_dense, S, tau_char_tanh)
    ax.plot(ta, ga, label=name, color=colors[name], lw=2)
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'recovered $\gamma(\tau)$ under tanh inverse decoder')
ax.set_title('γ-recovery signatures (clean traces)')
ax.legend(fontsize=9)
ax.set_ylim(0, 5)

plt.tight_layout()
fig.savefig(FIGURE_DIR / "00_envelope_shapes_recovered_gamma.png", dpi=150, bbox_inches="tight")
plt.show()




## 7.3 Primary Noise-Floor Scan: Five-Class Discriminability

This section tests how classification accuracy changes with noise level and trace length.

For each trace-length regime, the notebook generates noisy traces from all five envelope families, applies the tanh inverse decoder, extracts recovered $\gamma(\tau)$ features, and evaluates five-class classification accuracy. Chance performance is 20%.

The scan uses `n_per_class = 300` samples per family to keep runtime manageable. Larger runs produce the same qualitative pattern: discrimination remains strong at low-to-moderate noise and degrades as noise erases trajectory geometry.

This is the longest-running demonstration cell.


In [ ]:
NOISE_LEVELS = [0.0, 0.005, 0.01, 0.02, 0.04, 0.08, 0.16]
REGIMES = (
    ('short',    1.5, 12),
    ('medium',   3.0, 24),
    ('long',     6.0, 48),
)

def run_noise_scan(noise_model, n_per_class=300, seed=42):
    rng = np.random.default_rng(seed)
    specs = [(name, ENVELOPES[name], CALIB[name]) for name in ENVELOPES]
    results = {}
    for regime_name, tau_max, n_points in REGIMES:
        tau = np.linspace(0.01, tau_max, n_points)
        for noise in NOISE_LEVELS:
            r = run_discriminability(
                specs, tau, CALIB['tanh']['tau_char'],
                noise_model, noise, n_per_class, rng,
            )
            results[(regime_name, noise)] = r
    return results


results_het = run_noise_scan('heteroscedastic', seed=42)
results_gauss = run_noise_scan('gaussian', seed=43)

# Display as table
print("\nHETEROSCEDASTIC NOISE — 5-class accuracy (chance=0.20)\n")
header = f"{'regime':<10}" + "".join(f"σ={n:<7}" for n in NOISE_LEVELS)
print(header)
print("-" * len(header))
for regime_name, _, _ in REGIMES:
    row = f"{regime_name:<10}"
    for n in NOISE_LEVELS:
        r = results_het.get((regime_name, n))
        row += f"{r['accuracy']:<9.3f}" if r and 'accuracy' in r else f"{'--':<9}"
    print(row)

print("\nADDITIVE GAUSSIAN — 5-class accuracy\n")
print(header)
print("-" * len(header))
for regime_name, _, _ in REGIMES:
    row = f"{regime_name:<10}"
    for n in NOISE_LEVELS:
        r = results_gauss.get((regime_name, n))
        row += f"{r['accuracy']:<9.3f}" if r and 'accuracy' in r else f"{'--':<9}"
    print(row)



**What this shows.** Five-class accuracy remains above 80% through approximately $\sigma = 0.04$ across short, medium, and long trace regimes. Accuracy degrades sharply once noise approaches $\sigma = 0.08$, indicating that the limiting factor is loss of recoverable trajectory geometry rather than a failure of the classifier alone.

**Trace-length result.** Trace length has limited effect in this scan. Short, medium, and long regimes converge to similar noise floors. In this setup, the limiting constraint is measurement noise more than trace duration. That supports the use of short psycholinguistic traces, provided the relevant onset and curvature structure remains above the noise floor.


In [ ]:
# Plot noise floor curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, (results, title) in zip(
    axes,
    [(results_het, 'Heteroscedastic noise'),
     (results_gauss, 'Additive Gaussian noise')]
):
    for regime_name, _, _ in REGIMES:
        accs = [results.get((regime_name, n), {}).get('accuracy', np.nan)
                for n in NOISE_LEVELS]
        ax.plot(NOISE_LEVELS, accs, marker='o', lw=2, label=regime_name)
    ax.axhline(0.20, ls='--', color='gray', alpha=0.6, label='chance')
    ax.axhline(0.80, ls=':', color='red', alpha=0.6, label='80% threshold')
    ax.set_xscale('symlog', linthresh=0.005)
    ax.set_xlabel('noise σ')
    ax.set_ylabel('5-class classification accuracy')
    ax.set_title(title)
    ax.legend(loc='lower left', fontsize=9)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
fig.savefig(FIGURE_DIR / "01_noise_floor_curves.png", dpi=150, bbox_inches="tight")
plt.show()



## 7.4 Confusion matrix at the empirical noise regime

This section shows where classification errors concentrate at heteroscedastic noise $\sigma = 0.04$.

At this noise level, the calibrated envelope families separate into three patterns. Tanh, stretched exponential, and hyperbolic traces largely form distinct classes. Logistic and Gompertz are more often confused with each other.

For the spectral-admissibility test, the important question is not whether every non-tanh family separates cleanly from every other non-tanh family. The question is whether tanh remains distinguishable from bounded saturating foils after endpoint calibration and noise. In the calibrated baseline, it does.

The harder test comes next: whether a Gompertz foil can be tuned close enough to tanh that the recovered $\gamma(\tau)$ signature becomes indistinguishable at the declared noise floor.


In [ ]:
import matplotlib.pyplot as plt

key = ('short', 0.04)
r = results_het[key]
cm = r['confusion']
cm_norm = cm / cm.sum(axis=1, keepdims=True)
labels = r['labels']

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
ax.set_xlabel('predicted')
ax.set_ylabel('true')
ax.set_title(f"Short trace, σ=0.04 het. — accuracy {r['accuracy']:.3f}")
for i in range(len(labels)):
    for j in range(len(labels)):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax.text(j, i, f"{cm_norm[i,j]:.2f}", ha='center', va='center',
                color=color, fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
fig.savefig(FIGURE_DIR / "02_confusion_matrix_short_sigma004.png", dpi=150, bbox_inches="tight")
plt.show()




## 7.5 Pairwise Tanh-vs-Foil Discriminability

This section isolates the main spectral-admissibility question: how well tanh can be distinguished from each individual saturating foil.

Each test is binary. The discriminator compares tanh against one alternative family at a time, using the short-trace regime under heteroscedastic noise. This removes multi-class ambiguity and shows which foil is closest to the tanh retrieval geometry.


In [ ]:
alternatives = ['logistic', 'gompertz', 'stretched_exp', 'hyperbolic']
tau = np.linspace(0.01, 1.5, 12)  # short regime
tau_char = CALIB['tanh']['tau_char']

print(f"\n{'σ':<8}" + "".join(f"vs_{a:<14}" for a in alternatives))
print("-" * 72)
pairwise_results = {a: [] for a in alternatives}
for noise in NOISE_LEVELS:
    rng = np.random.default_rng(seed=11)
    row = f"{noise:<8}"
    for alt in alternatives:
        specs = [
            ('tanh', ENVELOPES['tanh'], CALIB['tanh']),
            (alt, ENVELOPES[alt], CALIB[alt]),
        ]
        r = run_discriminability(specs, tau, tau_char, 'heteroscedastic',
                                   noise, 400, rng)
        acc = r.get('accuracy', np.nan)
        pairwise_results[alt].append(acc)
        row += f"{acc:<17.3f}"
    print(row)



In [ ]:
# Plot pairwise curves
fig, ax = plt.subplots(figsize=(8, 5))
for alt in alternatives:
    ax.plot(NOISE_LEVELS, pairwise_results[alt], marker='o', lw=2,
             label=f'tanh vs {alt}', color=colors[alt])
ax.axhline(0.50, ls='--', color='gray', alpha=0.6, label='chance')
ax.axhline(0.80, ls=':', color='red', alpha=0.6, label='80% threshold')
ax.set_xscale('symlog', linthresh=0.005)
ax.set_xlabel('noise σ')
ax.set_ylabel('binary classification accuracy')
ax.set_title('Pairwise tanh vs alternative (short regime, heteroscedastic)')
ax.legend(loc='lower left', fontsize=9)
ax.set_ylim(0.4, 1.05)
plt.tight_layout()
fig.savefig(FIGURE_DIR / "03_pairwise_foil_discriminability.png", dpi=150, bbox_inches="tight")
plt.show()



**Reading.** All four foils remain distinguishable from tanh above 80% binary accuracy through approximately $\sigma = 0.05$. Gompertz is the closest alternative across the noise sweep. That makes it the relevant stress case for Section 7 and motivates the adversarial follow-up.


# 8. Adversarial Foil Follow-Up

The baseline tests use calibrated foils. They ask whether standard bounded saturating alternatives remain distinguishable from tanh after endpoint matching and noise.

Gompertz is the closest foil in those tests. This section therefore gives Gompertz the stronger position: its shape parameters $(c,\eta)$ are tuned specifically to minimize distance from the tanh trajectory.

The question is no longer whether a calibrated Gompertz curve is distinguishable. The question is whether a deliberately near-tanh Gompertz foil can erase the recovered $\gamma(\tau)$ signature at the declared noise floor.


## 8.1 Search for Adversarial Gompertz Foil

This section tunes the Gompertz shape parameters $(c,\eta)$ to make the Gompertz feature distribution as close as possible to the tanh feature distribution under matched noise.

The objective is the Mahalanobis distance between recovered $\gamma(\tau)$ feature distributions. The search uses differential evolution over a broad parameter range. Each objective evaluation is estimated by Monte Carlo sampling with 200 traces per class.

This is a stochastic adversarial search. Runtime is typically 20–30 seconds.


In [ ]:
def adversarial_gompertz_search(noise=0.04, tau_max=3.0, n_points=30,
                                  n_samples=200, seed=11):
    """Find Gompertz (c, eta) that maximally mimics tanh feature distribution."""
    rng_target = np.random.default_rng(seed)
    tau = np.linspace(0.01, tau_max, n_points)
    tau_char = CALIB['tanh']['tau_char']

    # Sample tanh feature distribution as the target
    tanh_feats = []
    for _ in range(500):
        S, _ = generate_trace(tanh_envelope, CALIB['tanh'], tau,
                                'heteroscedastic', noise, rng_target)
        ta, ga, _ = inverse_gamma(tau, S, tau_char)
        f = gamma_features_5(ta, ga)
        if not np.any(np.isnan(f)):
            tanh_feats.append(f)
    tanh_feats = np.array(tanh_feats)
    tanh_mean = tanh_feats.mean(axis=0)
    tanh_cov = np.cov(tanh_feats.T)
    tanh_cov_inv = np.linalg.pinv(tanh_cov + 1e-6 * np.eye(len(tanh_mean)))

    def objective(params):
        c, eta = params
        if eta <= 0:
            return 1e9
        rng_local = np.random.default_rng(42)
        feats = []
        for _ in range(n_samples):
            S, _ = generate_trace(gompertz_envelope, {'c': c, 'eta': eta},
                                    tau, 'heteroscedastic', noise, rng_local)
            ta, ga, _ = inverse_gamma(tau, S, tau_char)
            f = gamma_features_5(ta, ga)
            if not np.any(np.isnan(f)):
                feats.append(f)
        if len(feats) < 50:
            return 1e9
        g_mean = np.mean(feats, axis=0)
        diff = g_mean - tanh_mean
        return float(np.sqrt(diff @ tanh_cov_inv @ diff))

    result = differential_evolution(
        objective, bounds=[(-2.0, 2.0), (0.05, 5.0)],
        seed=7, maxiter=40, popsize=15, tol=1e-3, workers=1, polish=True,
    )
    return result.x, result.fun


print("Searching for adversarial Gompertz parameters...")
(c_adv, eta_adv), dist = adversarial_gompertz_search()
print(f"\nAdversarial Gompertz: c={c_adv:.4f}, eta={eta_adv:.4f}")
print(f"Mahalanobis distance to tanh: {dist:.3f}")



## 8.2 Diagnose the Adversarial Foil

This section checks what the adversarial search actually found.

The important question is whether the optimized Gompertz curve remains a meaningfully different trajectory that defeats the discriminator, or whether it becomes indistinguishable from tanh at the declared measurement resolution.

Three diagnostics are shown:

1. the adversarial Gompertz trajectory against the tanh trajectory;
2. pointwise deviation from tanh relative to the noise floor;
3. the recovered $\gamma(\tau)$ profile produced by the tanh inverse decoder.

If the adversarial curve differs from tanh below the noise floor, classifier failure is not evidence of model equivalence. It marks the resolution boundary of the test.


In [ ]:
tau_dense = np.linspace(0.001, 6.0, 400)
S_tanh = tanh_envelope(tau_dense, **CALIB['tanh'])
S_adv = gompertz_envelope(tau_dense, c=c_adv, eta=eta_adv)
S_calib = gompertz_envelope(tau_dense, **CALIB['gompertz'])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Left: trajectories
ax = axes[0]
ax.plot(tau_dense, S_tanh, 'b-', lw=2.5, label='tanh (extremal)')
ax.plot(tau_dense, S_adv, 'r--', lw=2,
         label=f'adversarial Gompertz\n(c={c_adv:.3f}, eta={eta_adv:.3f})')
ax.plot(tau_dense, S_calib, 'g:', lw=2, label='calibrated Gompertz')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'$S(\tau)$')
ax.set_title('Trajectory comparison')
ax.legend(fontsize=9, loc='lower right')

# Middle: deviations
ax = axes[1]
diff_adv = S_adv - S_tanh
diff_calib = S_calib - S_tanh
ax.plot(tau_dense, diff_adv, 'r-', lw=2, label='adv. Gompertz − tanh')
ax.plot(tau_dense, diff_calib, 'g--', lw=2, label='calib. Gompertz − tanh')
ax.axhline(0, color='k', lw=0.5)
ax.axhline(0.04, color='gray', lw=0.5, ls=':', label='noise floor σ=0.04')
ax.axhline(-0.04, color='gray', lw=0.5, ls=':')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel('deviation from tanh')
ax.set_title('Trajectory-level deviation')
ax.legend(fontsize=9)

# Right: γ recovery
ax = axes[2]
tau_char = CALIB['tanh']['tau_char']
for name, S, color, ls in [
    ('tanh', S_tanh, 'b', '-'),
    ('adv. Gompertz', S_adv, 'r', '--'),
    ('calib. Gompertz', S_calib, 'g', ':'),
]:
    ta, ga, _ = inverse_gamma(tau_dense, S, tau_char)
    ax.plot(ta, ga, color=color, ls=ls, lw=2.5 if ls=='-' else 2, label=name)
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'recovered $\gamma(\tau)$')
ax.set_title('γ-recovery (clean traces)')
ax.legend(fontsize=9)
ax.set_ylim(0, 4)

plt.tight_layout()
fig.savefig(FIGURE_DIR / "04_adversarial_gompertz_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()

# Numerical summary
rms_adv = np.sqrt(np.mean(diff_adv ** 2))
max_adv = np.max(np.abs(diff_adv))
rms_calib = np.sqrt(np.mean(diff_calib ** 2))
max_calib = np.max(np.abs(diff_calib))

print(f"\nTrajectory deviation from tanh:")
print(f"  Adversarial Gompertz: RMS={rms_adv:.5f}, max={max_adv:.5f}")
print(f"  Calibrated Gompertz:  RMS={rms_calib:.5f}, max={max_calib:.5f}")
print(f"  Noise floor σ:        0.04000")
print(f"  Adversarial max-deviation is {0.04/max_adv:.1f}x SMALLER than noise floor.")



This is the central result of the adversarial test. The search did not find a meaningfully different Gompertz trajectory that preserved a distinct shape while defeating the discriminator. It found a Gompertz parameterization whose trajectory is indistinguishable from tanh at the declared measurement resolution.

The trajectory deviation is roughly 2.5 times smaller than the noise floor, and the recovered $\gamma(\tau)$ profile tracks the tanh recovery profile closely.

The result should therefore be read as a resolution-boundary finding. A non-tanh bounded saturating envelope can evade detection only by approximating the tanh trajectory closely enough that the relevant geometric differences fall below the noise floor. Above that boundary, the trajectory geometry becomes recoverable again.


## 8.3 Classifier and Feature-Structure Robustness

This section tests whether the adversarial result depends on a particular classifier or on the compressed five-feature representation.

If the adversarial Gompertz remained distinguishable in a richer representation, then a more flexible classifier or a larger feature set should recover the separation. We therefore repeat the tanh-vs-adversarial-Gompertz test using five classifiers: LDA, QDA, SVM-RBF, Random Forest, and Gradient Boosting. The test uses the sixteen-feature representation defined above.

None of the classifiers restores strong discrimination. The best models remain near the low-60% accuracy range. This indicates that the failure is not caused by LDA or by the five-feature summary. It arises because the adversarial Gompertz trajectory has been tuned close enough to tanh that the relevant geometric differences fall below the declared noise floor.


In [ ]:
def collect_features(env_fn, env_params, tau, noise, n, rng,
                       featurizer):
    """Collect feature vectors over n trials."""
    X = []
    tau_char = CALIB['tanh']['tau_char']
    for _ in range(n):
        S, _ = generate_trace(env_fn, env_params, tau,
                                'heteroscedastic', noise, rng)
        if featurizer is gamma_features_5:
            ta, ga, _ = inverse_gamma(tau, S, tau_char)
            f = gamma_features_5(ta, ga)
        else:
            f = featurizer(tau, S, tau_char)
        if not np.any(np.isnan(f)):
            X.append(f)
    return np.array(X)


tau = np.linspace(0.01, 3.0, 30)
rng = np.random.default_rng(101)
N_PER = 600

X_tanh = collect_features(tanh_envelope, CALIB['tanh'], tau, 0.04, N_PER,
                            rng, gamma_features_16)
X_adv = collect_features(gompertz_envelope, {'c': c_adv, 'eta': eta_adv},
                           tau, 0.04, N_PER, rng, gamma_features_16)
X = np.vstack([X_tanh, X_adv])
y = np.array([0] * len(X_tanh) + [1] * len(X_adv))

classifiers = {
    'LDA': LinearDiscriminantAnalysis(),
    'QDA': QuadraticDiscriminantAnalysis(),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=0),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=0),
    'Gradient Boost': GradientBoostingClassifier(n_estimators=200, random_state=0),
}

print(f"\n{'classifier':<18}{'accuracy':<12}{'AUC':<10}")
print("-" * 40)
for name, clf in classifiers.items():
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    yp = cross_val_predict(clf, X, y, cv=cv)
    yp_proba = cross_val_predict(clf, X, y, cv=cv, method='predict_proba')
    acc = float(np.mean(yp == y))
    auc = float(roc_auc_score(y, yp_proba[:, 1]))
    print(f"{name:<18}{acc:<12.3f}{auc:<10.3f}")



# 9. Trajectory-RMS boundary characterization

Section 8 shows that an adversarial Gompertz foil becomes difficult to distinguish only when its trajectory approaches tanh below the declared noise floor. This section makes that boundary quantitative.

We sweep Gompertz parameter space, compute each trajectory’s RMS deviation from tanh, and evaluate tanh-vs-Gompertz discriminability as a function of that deviation.

The result is a trajectory-level resolution curve. For a given noise regime, it shows when an alternative envelope is distinguishable from tanh and when its deviation falls below the measurement resolution of the test. The boundary depends on trajectory geometry and noise level, not on the family label alone.


In [ ]:
def gompertz_trajectory_rms(c, eta, tau_dense=None):
    """RMS deviation of a Gompertz parameterization from calibrated tanh."""
    if tau_dense is None:
        tau_dense = np.linspace(0.001, 6.0, 400)
    S_tanh = tanh_envelope(tau_dense, **CALIB['tanh'])
    S_g = gompertz_envelope(tau_dense, c=c, eta=eta)
    return np.sqrt(np.mean((S_g - S_tanh) ** 2))


# Sweep through Gompertz parameter space, sampling at varying RMS distances
np.random.seed(MASTER_SEED)
candidate_pairs = []
for c in np.linspace(-1.5, 1.5, 12):
    for eta in np.linspace(0.1, 2.0, 12):
        rms = gompertz_trajectory_rms(c, eta)
        candidate_pairs.append((c, eta, rms))

# Bin by RMS and pick representatives
candidate_pairs = sorted(candidate_pairs, key=lambda x: x[2])
rms_bins = [(0.0, 0.005), (0.005, 0.01), (0.01, 0.02), (0.02, 0.04),
             (0.04, 0.08), (0.08, 0.15), (0.15, 0.30)]
representatives = []
for low, high in rms_bins:
    in_bin = [p for p in candidate_pairs if low <= p[2] < high]
    if in_bin:
        # Take the one closest to bin midpoint
        mid = (low + high) / 2
        rep = min(in_bin, key=lambda p: abs(p[2] - mid))
        representatives.append(rep)

print("Representative Gompertz parameterizations by trajectory RMS distance from tanh:")
print(f"{'c':<10}{'eta':<10}{'RMS':<10}")
for c, eta, rms in representatives:
    print(f"{c:<10.3f}{eta:<10.3f}{rms:<10.5f}")



In [ ]:
# For each representative, compute discriminability vs tanh at σ=0.04
tau = np.linspace(0.01, 3.0, 30)
tau_char = CALIB['tanh']['tau_char']

print(f"\nDiscriminability tanh vs Gompertz at σ=0.04 heteroscedastic:")
print(f"{'RMS dist':<12}{'(c, eta)':<22}{'accuracy':<12}{'AUC':<10}")
print("-" * 56)

boundary_data = []
for c, eta, rms in representatives:
    rng = np.random.default_rng(seed=int(rms * 1e6) % 1000)
    specs = [
        ('tanh', tanh_envelope, CALIB['tanh']),
        ('gompertz', gompertz_envelope, {'c': c, 'eta': eta}),
    ]
    r = run_discriminability(specs, tau, tau_char, 'heteroscedastic',
                               0.04, 500, rng)
    acc = r['accuracy']
    auc = r.get('auc', np.nan)
    boundary_data.append((rms, acc, auc, c, eta))
    print(f"{rms:<12.5f}({c:.3f}, {eta:.3f})        {acc:<12.3f}{auc:<10.3f}")



In [ ]:
# Plot the boundary curve
fig, ax = plt.subplots(figsize=(8, 5))
rmss = [b[0] for b in boundary_data]
accs = [b[1] for b in boundary_data]
aucs = [b[2] for b in boundary_data]

ax.plot(rmss, accs, 'o-', lw=2, label='binary accuracy', color='C0')
ax.plot(rmss, aucs, 's-', lw=2, label='ROC AUC', color='C2')
ax.axhline(0.5, ls='--', color='gray', alpha=0.6, label='chance')
ax.axhline(0.8, ls=':', color='red', alpha=0.6, label='80% threshold')
ax.axvline(0.04, ls=':', color='black', alpha=0.5, label='noise floor σ=0.04')
ax.set_xscale('log')
ax.set_xlabel('trajectory RMS deviation from tanh')
ax.set_ylabel('discriminability')
ax.set_title('Empirical distinguishability vs trajectory deviation\n(σ=0.04 heteroscedastic)')
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0.4, 1.05)
plt.tight_layout()
fig.savefig(FIGURE_DIR / "05_rms_boundary_curve.png", dpi=150, bbox_inches="tight")
plt.show()



**Resulting claim.** A non-tanh bounded saturating envelope whose trajectory RMS distance from tanh exceeds the declared noise floor is distinguishable from tanh under the framework’s own inverse decoder. When the trajectory deviation falls below the noise floor, the curves are not reliably separable at that measurement resolution.

This is the operational form of the analytic-speed-limit conditional. The theory states that, under spectral admissibility, tanh is the unique extremal saturator. The curve above shows how that conditional appears under finite resolution: distinguishability is governed by trajectory deviation relative to noise, not by the envelope-family label alone.


## Summary

The spectral-admissibility condition has measurement-level consequences in this synthetic setting.

Under the noise regimes tested here:

* The tanh smooth-access baseline produces a recovered $\gamma(\tau)$ signature that remains distinguishable from logistic, calibrated Gompertz, stretched-exponential, and hyperbolic envelopes through approximately $\sigma = 0.04$, with five-class accuracy above 80%.
* Trace length is not the binding constraint in these simulations. Noise is. Short traces remain informative when onset and curvature structure stay above the measurement floor.
* Gompertz is the closest foil. Logistic, stretched-exponential, and hyperbolic alternatives separate more cleanly.
* Adversarial Gompertz parameterizations defeat the discriminator only when they approximate the tanh trajectory closely enough that their deviations fall below the noise floor.
* Discriminability tracks trajectory-level RMS deviation from tanh, not envelope-family labels.

**What this is not.** This is not a human empirical validation result. It is a synthetic-trace test of the inverse decoder’s discriminative power. To convert this into an empirical claim about ERP, self-paced reading, or eye-tracking data, the noise model must be calibrated against residual structure in locked empirical pipelines.

**Reproducibility.** All stochastic routines use explicit seeds. Running the notebook end to end with a standard scientific Python stack reproduces the figures and summary tables above.


## 10. Artifact Export and Verification Report

This section exports compact CSV/JSON artifacts and maps each theory-side claim to a PASS/CHECK status. The thresholds are declared near the top of the notebook. A PASS means the numerical result meets the frozen verification criterion for `LINGUISTIC_SPECTRAL_ADMISSIBILITY_V1`.


In [ ]:
import pandas as pd

# Calibration artifact.
calibration_rows = []
for name, params in CALIB.items():
    s1 = float(ENVELOPES[name](1.0, **params))
    s3 = float(ENVELOPES[name](3.0, **params))
    calibration_rows.append({
        "envelope": name,
        "parameters": json.dumps({k: float(v) for k, v in params.items()}),
        "S_at_1": s1,
        "S_at_3": s3,
        "abs_error_at_3": abs(s3 - 0.95),
    })
calibration_df = pd.DataFrame(calibration_rows)
calibration_df.to_csv(OUTPUT_DIR / "spectral_calibration_summary.csv", index=False)

# Noise-scan artifacts.
def _noise_scan_to_df(results, noise_model):
    rows = []
    for (regime, noise), result in results.items():
        rows.append({
            "noise_model": noise_model,
            "regime": regime,
            "noise_sigma": noise,
            "accuracy": result.get("accuracy", np.nan),
            "n_used": result.get("n_used", np.nan),
            "n_failed": result.get("n_failed", np.nan),
        })
    return pd.DataFrame(rows)

noise_scan_df = pd.concat([
    _noise_scan_to_df(results_het, "heteroscedastic"),
    _noise_scan_to_df(results_gauss, "gaussian"),
], ignore_index=True)
noise_scan_df.to_csv(OUTPUT_DIR / "spectral_noise_scan.csv", index=False)

# Pairwise artifact.
pairwise_rows = []
for alt, accs in pairwise_results.items():
    for noise, acc in zip(NOISE_LEVELS, accs):
        pairwise_rows.append({"foil": alt, "noise_sigma": noise, "accuracy": acc})
pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_df.to_csv(OUTPUT_DIR / "spectral_pairwise_foil_discriminability.csv", index=False)

# Adversarial classifier artifact.
adversarial_classifier_rows = []
for name, clf in classifiers.items():
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    yp = cross_val_predict(clf, X, y, cv=cv)
    yp_proba = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")
    adversarial_classifier_rows.append({
        "classifier": name,
        "accuracy": float(np.mean(yp == y)),
        "auc": float(roc_auc_score(y, yp_proba[:, 1])),
    })
adversarial_classifier_df = pd.DataFrame(adversarial_classifier_rows)
adversarial_classifier_df.to_csv(OUTPUT_DIR / "spectral_adversarial_classifier_robustness.csv", index=False)

# Boundary artifact.
boundary_df = pd.DataFrame(boundary_data, columns=["rms_distance", "accuracy", "auc", "c", "eta"])
boundary_df.to_csv(OUTPUT_DIR / "spectral_rms_boundary.csv", index=False)

print(f"Wrote verification artifacts to {OUTPUT_DIR}")


In [ ]:
def _pass(ok):
    return "PASS" if bool(ok) else "CHECK"

calibration_error_max = float(calibration_df["abs_error_at_3"].max())
primary_ref = noise_scan_df[(noise_scan_df["noise_model"] == PRIMARY_NOISE_MODEL) &
                            (np.isclose(noise_scan_df["noise_sigma"], REFERENCE_NOISE))]
primary_min_acc = float(primary_ref["accuracy"].min())

pairwise_ref = pairwise_df[np.isclose(pairwise_df["noise_sigma"], REFERENCE_NOISE)]
pairwise_min_acc = float(pairwise_ref["accuracy"].min())

adversarial_rms = float(gompertz_trajectory_rms(c_adv, eta_adv))
adversarial_max_dev = float(np.max(np.abs(
    gompertz_envelope(np.linspace(0.001, 6.0, 400), c=c_adv, eta=eta_adv) -
    tanh_envelope(np.linspace(0.001, 6.0, 400), **CALIB["tanh"])
)))
adversarial_max_classifier_acc = float(adversarial_classifier_df["accuracy"].max())

boundary_above_floor = boundary_df[boundary_df["rms_distance"] >= REFERENCE_NOISE]
boundary_min_acc_above_floor = float(boundary_above_floor["accuracy"].min()) if len(boundary_above_floor) else np.nan

SPECTRAL_VERIFICATION_REPORT = pd.DataFrame([
    {
        "claim": "Common saturation calibration",
        "artifact": "spectral_calibration_summary.csv",
        "result": f"max |S(3)-0.95| = {calibration_error_max:.2e}",
        "status": _pass(calibration_error_max <= PASS_THRESHOLDS["calibration_abs_error_max"]),
    },
    {
        "claim": "Primary five-class finite-resolution discriminability",
        "artifact": "spectral_noise_scan.csv; 01_noise_floor_curves.png",
        "result": f"min {PRIMARY_NOISE_MODEL} accuracy at sigma={REFERENCE_NOISE:.2f} = {primary_min_acc:.3f}",
        "status": _pass(primary_min_acc >= PASS_THRESHOLDS["five_class_accuracy_min_at_reference_noise"]),
    },
    {
        "claim": "Pairwise calibrated-foil discriminability",
        "artifact": "spectral_pairwise_foil_discriminability.csv; 03_pairwise_foil_discriminability.png",
        "result": f"min tanh-vs-foil accuracy at sigma={REFERENCE_NOISE:.2f} = {pairwise_min_acc:.3f}",
        "status": _pass(pairwise_min_acc >= PASS_THRESHOLDS["pairwise_accuracy_min_at_reference_noise"]),
    },
    {
        "claim": "Adversarial Gompertz near-equivalence",
        "artifact": "04_adversarial_gompertz_diagnostic.png",
        "result": f"max deviation = {adversarial_max_dev:.3f}; RMS = {adversarial_rms:.3f}; noise floor = {REFERENCE_NOISE:.3f}",
        "status": _pass(adversarial_max_dev <= PASS_THRESHOLDS["adversarial_max_deviation_max"]),
    },
    {
        "claim": "Classifier robustness under near-equivalent adversarial foil",
        "artifact": "spectral_adversarial_classifier_robustness.csv",
        "result": f"max classifier accuracy = {adversarial_max_classifier_acc:.3f}",
        "status": _pass(adversarial_max_classifier_acc <= PASS_THRESHOLDS["adversarial_classifier_accuracy_max"]),
    },
    {
        "claim": "Trajectory-RMS boundary",
        "artifact": "spectral_rms_boundary.csv; 05_rms_boundary_curve.png",
        "result": f"min accuracy above RMS noise floor = {boundary_min_acc_above_floor:.3f}",
        "status": _pass(boundary_min_acc_above_floor >= PASS_THRESHOLDS["rms_boundary_accuracy_min_above_noise_floor"]),
    },
])

SPECTRAL_VERIFICATION_REPORT.to_csv(OUTPUT_DIR / "spectral_verification_report.csv", index=False)
print(f"Analysis preset: {ANALYSIS_PRESET}")
display(SPECTRAL_VERIFICATION_REPORT)
if (SPECTRAL_VERIFICATION_REPORT["status"] == "PASS").all():
    print("Verification report complete: available artifacts match expected theory-side checks.")
else:
    print("Review required: one or more spectral verification checks are outside expected tolerance.")


In [ ]:
validation_manifest = {
    "artifact": ARTIFACT_NAME,
    "version": ARTIFACT_VERSION,
    "analysis_preset": ANALYSIS_PRESET,
    "run_mode": RUN_MODE,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "master_seed": MASTER_SEED,
    "primary_noise_model": PRIMARY_NOISE_MODEL,
    "reference_noise_sigma": REFERENCE_NOISE,
    "noise_levels": [float(x) for x in NOISE_LEVELS],
    "trace_regimes": [{"name": name, "tau_max": tau_max, "n_points": n_points}
                      for name, tau_max, n_points in REGIMES],
    "samples": {
        "five_class_per_class": 300,
        "pairwise_per_class": 400,
        "boundary_per_class": 500,
        "adversarial_classifier_per_class": int(N_PER),
    },
    "pass_thresholds": PASS_THRESHOLDS,
    "envelopes": list(ENVELOPES.keys()),
    "adversarial_gompertz": {"c": float(c_adv), "eta": float(eta_adv)},
    "verification_status": SPECTRAL_VERIFICATION_REPORT.to_dict(orient="records"),
    "artifacts": [
        "spectral_calibration_summary.csv",
        "spectral_noise_scan.csv",
        "spectral_pairwise_foil_discriminability.csv",
        "spectral_adversarial_classifier_robustness.csv",
        "spectral_rms_boundary.csv",
        "spectral_verification_report.csv",
        "figures/00_envelope_shapes_recovered_gamma.png",
        "figures/01_noise_floor_curves.png",
        "figures/02_confusion_matrix_short_sigma004.png",
        "figures/03_pairwise_foil_discriminability.png",
        "figures/04_adversarial_gompertz_diagnostic.png",
        "figures/05_rms_boundary_curve.png",
    ],
}
with open(OUTPUT_DIR / "validation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(validation_manifest, f, indent=2)
print(f"Wrote manifest: {OUTPUT_DIR / 'validation_manifest.json'}")

